# Data choice and preprocessing (citation attribution)

Dans la littérature, on trouve de nombreux datasets conçus pour tester la qualité et fiabilité des choix de sources pour prouver une réponse dans des systèmes RAG.

Pour le moment, j'ai sélectionné 5 datasets :
- [MuSiQue](https://arxiv.org/pdf/2108.00573)
- [HotPotQA](https://arxiv.org/pdf/1809.09600)
- [GaRAGe](https://arxiv.org/pdf/2506.07671)
- [QAMPARI](https://arxiv.org/pdf/2305.14627) (sous ensemble du dataset ALCE)
- [RAGBench](https://arxiv.org/pdf/2407.11005)

De base, ils ont des structures très variées. J'ai choisi d'avoir un format uniformisé pour pouvoir appliquer les mêmes méthodes dessus.

Les traitements ont été adaptés à chaque dataset et précisés dans leur sous partie. A l'issue des modifications apportées, tous les datasets suivent ce format :    

- **id** (str) : identifiant unique de chaque sample
- **question** (str) : query
- **contexts** {"id": Int, "text": String} : ensemble des documents (chunks retrieved)
- **gold_ids** (liste de int) : liste des identifiants des documents à citer (les passages pertinents)
- **answer** (str) : réponse

Ce format a été motivé par :    
- la simplicité de gestion pour faire le lien entre un chunk et son identifiant et les chunks golds
- la facilité de calcul des métriques quantitatives  


Les données sont enregistrées dans des fichiers.parquet

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
path = "/content/drive/MyDrive/MVA/Stage"

In [ ]:
import pandas as pd
import random

## MuSiQue  

Ce dataset a pour but d'évaluer la robutesse des systèmes face au raisonnement multi-hop : la réponse ne découle pas directement d'un (ou plusieurs) document mais d'une succession d'étapes impliquant entre 2 et 4 documents. Par exemple, le chunk A ne va pas forcément répondre à la question mais le chunk B qui lui répond directement, a besoin de connaître le chunk A. Le chunk A doit donc être mentionné dans les sources.  


Le dataset contient pour chaque question 20 documents :    
- 2 à 4 golds chunks
- 16 à 20 distracteurs  

Le dataset a été créé en se basant sur des ensembles de paires Q/A "single-hop" et ont construit à partir de là un des questions "multi-hop". Les données d'origine proviennent de pages Wikipedia en anglais.

In [ ]:
from datasets import load_dataset

ds = load_dataset("dgslibisey/MuSiQue", split="train[:3000]")

musique_ans_v1.0_train.jsonl:   0%|          | 0.00/241M [00:00<?, ?B/s]

musique_ans_v1.0_dev.jsonl:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19938 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2417 [00:00<?, ? examples/s]

In [ ]:
def custom_musique(raw_dataset):
    simplified_data = []

    for item in raw_dataset:
        # extrait les documents et incrémente un id pour chaque doc
        contexts = [ {"id": p['idx'], "text": p['paragraph_text']} for p in item['paragraphs'] ]

        # on recupere les id des docs considéré comme pertinents
        gold_ids = [p['idx'] for p in item['paragraphs'] if p['is_supporting']]

        simplified_data.append({
            "id": item['id'],
            "question": item['question'],
            "contexts": contexts,
            "gold_ids": gold_ids,
            "answer": item['answer']
        })
    df = pd.DataFrame(simplified_data)
    return df



In [ ]:
df_musique = custom_musique(ds)

In [ ]:
df_musique_1000 = df_musique.sample(n=1000, random_state=42)
df_musique_1000.to_parquet(f"{path}/data/musique.parquet", index=False)

## HotPotQA

Les données vienennt de Wikipedia en anglais. Les auteurs ont utilisé les 1ers paragraphes (introduction) des articles comme matière première.

Ce dataset a été conçu pour gérer le raisonnement multi-hop également.

Pour le raisonnement multi-hop, ils partaient de 2 articles : ils ont extrait un premier paragraphe, puis ont suivi un hyperlien présent dans ce texte pour sélectionner le paragraphe d'un second article connexe. Des annotateurs humains ensuite devaient formuler des questions qui nécessitent les 2 paragraphes.

Le dataset contient :     
- 2 golds chunk (les 2 paragraphes)
- les faux positifs : ils ont pris la question générée par l'humain et l'ont tapée dans un moteur de recherche interne (basé sur TF IDF). Il y en a 8 par question (ceux avec les scores les plus élevés)




Dans les données brutes, les paragraphes sont segmentés (sentence level). On commence par concaténer les différentes phrase d'un même paragraphe ensemble.  

Pour trouver les gold_ids, il faut regarder la colonne "supporting_facts" du dataset original : il contient les titres des paragraphes à citer. Pour établir nos gold_ids, on fait un exact match sur le titre, qui est aussi dans la colonne "context" du dataset de base et permet de faire l'identification des passages pertinents.

In [ ]:
from datasets import load_dataset

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="train[:3000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [ ]:
ds[0]

{'id': '5a7a06935542990198eaf050',
 'question': "Which magazine was started first Arthur's Magazine or First for Women?",
 'answer': "Arthur's Magazine",
 'type': 'comparison',
 'level': 'medium',
 'supporting_facts': {'title': ["Arthur's Magazine", 'First for Women'],
  'sent_id': [0, 0]},
 'context': {'title': ['Radio City (Indian radio station)',
   'History of Albanian football',
   'Echosmith',
   "Women's colleges in the Southern United States",
   'First Arthur County Courthouse and Jail',
   "Arthur's Magazine",
   '2014–15 Ukrainian Hockey Championship',
   'First for Women',
   'Freeway Complex Fire',
   'William Rast'],
  'sentences': [["Radio City is India's first private FM radio station and was started on 3 July 2001.",
    ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).',
    ' It plays Hindi, English and regional songs.',
    ' It was launch

In [ ]:
def custom_hotpot(raw_dataset):
    simplified_data = []

    for item in raw_dataset:
        # le titre est l'élément dans le dataset qui permet d'identifier les gold chunks
        all_titles = item['context']['title']
        all_sentences_lists = item['context']['sentences']

        # creation "ideale" du context (séparé par phrase de base)
        contexts = []
        for i, sentences in enumerate(all_sentences_lists):
            # on concatène toutes les phrases du document
            full_text = " ".join([s.strip() for s in sentences])
            contexts.append({
                "id": i,
                "text": full_text
            })

        # recupere les gold ids
        gold_ids = []
        gold_titles_unique = set(item['supporting_facts']['title'])

        for gold_title in gold_titles_unique:
            if gold_title in all_titles:
                gold_id = all_titles.index(gold_title)
                gold_ids.append(gold_id)

        # create dataframe
        simplified_data.append({
            "id": item['id'],
            "question": item['question'],
            "contexts": contexts,
            "gold_ids": sorted(gold_ids),
            "answer": item['answer']
        })

    return pd.DataFrame(simplified_data)


In [ ]:
df_hotpot = custom_hotpot(ds)

In [ ]:
df_hotpot.head(20)

,id,question,contexts,gold_ids,answer
0,5a7a06935542990198eaf050,Which magazine was started first Arthur's Maga...,"[{'id': 0, 'text': 'Radio City is India's firs...","[5, 7]",Arthur's Magazine
1,5a879ab05542996e4f30887e,The Oberoi family is part of a hotel company t...,"[{'id': 0, 'text': 'The Ritz-Carlton Jakarta i...","[1, 6]",Delhi
2,5a8d7341554299441c6b9fe5,Musician and satirist Allie Goertz wrote a son...,"[{'id': 0, 'text': 'Lisa Marie Simpson is a fi...","[3, 4]",President Richard Nixon
3,5a82171f5542990a1d231f4a,What nationality was James Henry Miller's wife?,"[{'id': 0, 'text': 'Moloch: or, This Gentile W...","[4, 5]",American
4,5a84dd955542997b5ce3ff79,Cadmium Chloride is slightly soluble in this c...,"[{'id': 0, 'text': 'Cadmium chloride is a whit...","[0, 5]",alcohol
5,5a7e36045542991319bc9440,Which tennis player won more Grand Slam titles...,"[{'id': 0, 'text': 'Li Na (; ; born 26 Februar...","[2, 5]",Jonathan Stark
6,5adf44985542993a75d2646d,Which genus of moth in the world's seventh-lar...,"[{'id': 0, 'text': 'India, officially the Repu...","[0, 7]",Crambidae
7,5a832c3455429954d2e2ec41,Who was once considered the best kick boxer in...,"[{'id': 0, 'text': 'The 1998 Verano de Escánda...","[4, 6]",Badr Hari
8,5a7d0db955429909bec76924,"The Dutch-Belgian television series that ""Hous...","[{'id': 0, 'text': 'House of Anubis is a myste...","[0, 8]",2006
9,5a89372855429951533612e6,What is the length of the track where the 2013...,"[{'id': 0, 'text': 'Mount Panorama Circuit is ...","[0, 6]",6.213 km long


In [ ]:
df_hotpot_1000 = df_hotpot.sample(n=1000, random_state=42)

In [ ]:
df_hotpot_1000.to_parquet(f"{path}/data/hotpot.parquet", index=False)

## GaRAGe

Les données proviennent de plusieurs sources : Wikipedia, Arxiv et d'autres pages de documentation techniques.

Il incarne un vrai système RAG : les documents sont récupérés via un retriever. Un annotateur humain lit la question, rédige la réponse et choisit les sources considérées comme "golds chunks".  

Le dataset original catégorise les questions selon leur type. J'ai choisi de garder les types : simple, simple avec condition et comparaison. Pour déterminer les gold ids, on se base sur le jugement humain : une colonne dans la réponse binaire "yes" / "no" pour dire si l'humain cite ou pas un document.

In [ ]:
def custom_garage(raw_dataset):
    simplified_data = []

    # on choisit que ces types de questions
    allowed_complexities = ["Simple", "Simple w. condition", "Comparison"]

    for item in raw_dataset:
        # on filtre pour  les 3 complexités qu'on veut
        if item.get('question_complexity') not in allowed_complexities:
            continue

        # creation "ideale" du contexte
        contexts = []
        for i, doc in enumerate(item['grounding']):
            text_content = ""
            # On cherche dynamiquement la clé 'cite_X' qui n'est pas vide
            for key, value in doc.items():
                if key.startswith('cite_') and value is not None:
                    text_content = value.strip()
                    break # On a trouvé le texte, on passe au document suivant

            contexts.append({
                "id": i,
                "text": text_content
            })

        # récupération des gold id en vérifiant le choix humain (evidence_cited)
        gold_ids = []
        for i, cited_tag in enumerate(item['evidence_cited']):
            if cited_tag == "YES":
                gold_ids.append(i)

        # create dataframe
        simplified_data.append({
            "id": item['sample_id'],
            "question": item['question'],
            "contexts": contexts,
            "gold_ids": gold_ids,
            "answer": item['answer_generate'] # La réponse rédigée par l'annotateur
        })

    return pd.DataFrame(simplified_data)


In [ ]:
ds_garage = load_dataset("AmazonScience/GaRAGe", split="train") # Ajuste le split si besoin

README.md: 0.00B [00:00, ?B/s]

data/GaRAGe_benchmark.jsonl:   0%|          | 0.00/28.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2366 [00:00<?, ? examples/s]

In [ ]:
df_garage = custom_garage(ds_garage)

In [ ]:
df_garage.head(10)

,id,question,contexts,gold_ids,answer
0,3e85c5e3-ac81-484f-bee3-9cba1000169f,What strategic move did BlackRock make in 2024...,"[{'id': 0, 'text': 'In 2024, BlackRock spent a...","[2, 3, 4, 5, 6, 7, 8, 9, 11, 12]","In 2024, BlackRock took several strategic step..."
1,340106be-dc22-4c74-9c74-620bd4246d75,How effective were Adams Resources' hedging st...,"[{'id': 0, 'text': 'The company announced its ...","[0, 1, 2, 7]",Adams Resources & Energy's hedging strategies ...
2,2d38b3a4-fe6e-4850-aea9-92e4832d00ef,What impact will 2024's CRISPR advancements ha...,"[{'id': 0, 'text': 'As our understanding of th...","[1, 3, 6]",The advancements in CRISPR technology in 2024 ...
3,93d93bdb-de21-4f0b-b91c-1fd6c98911e2,How do the circulating variants KP.3.1.1 and X...,"[{'id': 0, 'text': 'Abstract KP.3.1.1 has surp...","[0, 5]",The circulating variants KP.3.1.1 and XEC diff...
4,f4e5287e-b71e-437b-82e7-3997ab839220,What role does Explainable AI play in mitigati...,"[{'id': 0, 'text': 'The Role of Explainable AI...","[0, 2, 3, 4, 5, 6, 7, 10, 11, 14]",Explainable AI (XAI) plays a critical role in ...
5,7dc8aab3-a684-46c1-bb8c-200ce3f92f0e,What is the relationship between a country's E...,"[{'id': 0, 'text': 'Indeed we nd the ECI has ...",[2],There is a strong relationship between a count...
6,ae2f55e8-6e75-4223-909d-9585c07f52f2,What are the key differences between tradition...,"[{'id': 0, 'text': 'This means that Intel Opta...","[0, 2, 4, 6, 7, 10, 11, 13]",The key differences between traditional SSDs a...
7,d336261c-90cc-432d-bcbe-18b076a49e9a,What new permissions are required for Google C...,"[{'id': 0, 'text': 'Permissions and roles. Wit...",[],There is not enough grounding for an answer.
8,856021b6-861b-4b16-bc8a-9c1439d220aa,How does safe RL integrate safety constraints ...,"[{'id': 0, 'text': 'safe reinforcement learnin...","[2, 3, 4, 6, 7, 13]",Safe reinforcement learning (Safe RL) integrat...
9,467f3ec7-807a-440f-89ab-3645015c7ec0,What advancements in numerical methods improve...,"[{'id': 0, 'text': 'Abstract. The nonlinear Sc...","[0, 9]",Advancements in numerical methods for simulati...


In [ ]:
df_garage.shape

(624, 5)

In [ ]:
df_garage.to_parquet(f"{path}/data/garage.parquet", index=False)

## QAMPARI


QAMPARI provient du benchmark ALCE (Automatic LLMs' Citation Evaluation). Le benchmark a été conçu pour évaluer la capacité des LLMs à générer des réponses exactes tout en citant correctement leurs sources dans un contexte RAG. De nombreux papiers le mentionnent pour leur test ou pour du transfer learning.

Il incarne aussi un système RAG réel : les documents du contexte ont été choisis par un retriever / rerankers (DPR, BM25, Oracle reranker).  

Il contient 3 datasets : ASQA, QAMPARI et ELI5. J'ai choisi QAMPARI pour le moment avec comme retriever : DPR. Les données viennent aussi de Wikipedia.

Pour chaque question, ils ont récupéré 100 documents classés selon leur score de smilarité. QAMPARI est un dataset de type questions à choix multiple : les réponses sont listées.

Le dataset original contient uun attribution "has_answer" booléen qui permet d'identifier les chunks pertinents. J'utilise cette variable pour traduire les true en gold _ids. Pour le moment j'ai choisi de conserver 40 documents par question. En priorité, je sélectionne tous les documents pertinents et complète le quota avec les faux négatifs. Ces derniers sont choisis selon leur classement : je prends ceux aux scores de similarité les plus élevés.

In [ ]:
import pandas as pd
import random

def custom_qampari(raw_dataset, max_docs=40):
    simplified_data = []

    for item in raw_dataset:
        # separation des gold ids et des distracteurs en se basant sur la variable has_answer
        gold_docs = [doc for doc in item['docs'] if doc.get('has_answer') == True]
        distractor_docs = [doc for doc in item['docs'] if doc.get('has_answer') == False]

        # pour ce dataset, je choisis de garder les samples qui ont au moins un doc pertinent
        if not gold_docs:
            continue

        # en 1) on sélectionne tous les docs true
        selected_golds = gold_docs[:max_docs]

        # calcul du nb de faux positifs par question pour = seuil
        num_distractors_needed = max_docs - len(selected_golds)

        #en 2) on complete pour atteindre le max (40) avec les top distracteurs
        selected_distractors = distractor_docs[:num_distractors_needed]

        # on assemble et on shuffle pour éviter un biais
        all_selected_docs = selected_golds + selected_distractors
        random.seed(42)
        random.shuffle(all_selected_docs)

        # creation de nos contexts et gold ids
        contexts = []
        gold_ids = []

        for i, doc in enumerate(all_selected_docs):
            contexts.append({
                "id": i,
                "text": doc['text']
            })
            # Si le document fait partie de ceux qui ont la réponse, on garde son nouvel id
            if doc['has_answer']:
                gold_ids.append(i)

        #create du dataframe
        simplified_data.append({
            "id": item['id'],
            "question": item['question'],
            "contexts": contexts,
            "gold_ids": sorted(gold_ids),
            "answer": item['answer']
        })

    return pd.DataFrame(simplified_data)

In [ ]:
import json

with open(f"{path}/qampari_eval_dpr_top100.json", "r") as f:
  ds_qampari = json.load(f)

In [ ]:
df_qampari = custom_qampari(ds_qampari, max_docs=40)

In [ ]:
df_qampari.head(20)

,id,question,contexts,gold_ids,answer
0,136__wikidata_intersection__dev,Harmony Korine was both screenwriter and direc...,"[{'id': 0, 'text': 'Amy Goldstein Amy Goldstei...","[1, 28, 29, 38]","Spring Breakers, Trash Humpers, Julien Donkey-..."
1,366__wikidata_comp__dev,Who directed a film that had P. Balachandran a...,"[{'id': 0, 'text': 'T. S. B. K. Moulee T. S. B...","[0, 1, 8, 28, 29, 32, 33, 34, 38, 39]","Kamal, P. Balachandran, T. K. Rajeev Kumar, V...."
2,663__wikidata_simple__dev,The Russian Empire has what ships registered t...,"[{'id': 0, 'text': 'The Victoria Cross, Britai...","[0, 1, 2, 7, 8, 14, 28, 29, 32, 33, 34, 38, 39]","Russian gunboat Sivuch, Russian cruiser Admira..."
3,777__wikidata_simple__dev,What player was selected in the draft by the S...,"[{'id': 0, 'text': '2016 NWHL Draft The 2016 N...","[28, 29, 38]","Tianna Hawkins, Ashley Walker, Barbara Turner,..."
4,386__wikidata_comp__dev,Where was a Bishop of Bradford taught?,"[{'id': 0, 'text': 'since been canonised or be...","[1, 28, 29, 38]","Nottingham High School, Marlborough College, K..."
5,801__wikidata_simple__dev,What piece of literature did David G. Hartwell...,"[{'id': 0, 'text': 'hire a 27-year-old to be e...",[29],"Year's Best SF 5, Year's Best SF 11, Year's Be..."
6,8__wikitables_composition__dev,Which of upstate New York's tallest buildings ...,"[{'id': 0, 'text': 'Tower, the observation dec...","[1, 8, 28, 29, 32, 33, 38, 39]","Seneca One Tower, Buffalo City Hall, Rand Buil..."
7,481__wikitables_simple__dev,What magazine is a science fiction magazine ?,"[{'id': 0, 'text': 'Jack Williamson for exampl...","[0, 1, 2, 4, 5, 7, 8, 10, 12, 13, 14, 17, 18, ...","Amazing Stories, Analog Science Fiction and Fa..."
8,439__wikidata_intersection__dev,For which movie did Mani Ratnam work on the sc...,"[{'id': 0, 'text': 'Ramayana"". Three years lat...","[0, 1, 2, 7, 8, 14, 18, 28, 29, 32, 33, 34, 35...","Dil Se.., Thiruda Thiruda, Raavan, O Kadhal Ka..."
9,772__wikidata_simple__dev,What is an example of the marine vessel classi...,"[{'id': 0, 'text': '1945, the quadruple 2 cm m...","[1, 28, 29, 38]","SM U-31, SM U-33, SM U-37, SM U-41, SM U-38, S..."


In [ ]:
gold_ids = df_qampari.loc[
    df_qampari["id"] == "439__wikidata_intersection__dev",
    "gold_ids"
].iloc[0]

print(gold_ids)

[0, 1, 2, 7, 8, 14, 18, 28, 29, 32, 33, 34, 35, 36, 38, 39]


In [ ]:
df_qampari.shape

(705, 5)

In [ ]:
df_qampari.to_parquet(f"{path}/data/qampari.parquet", index=False)

## RAGBench

RAGBench est aussi un benchmark qui inclut 12 dataset simulant un RAG. Les auteur les ont retraités avec des LLMs et des annotateurs humains pour les standardiser sous un format spécifique à l'évaluation RAG,. Un point important de cet ensemble de datasets et la tracabilité qui est mise en évidence (intéressant pour tester le raisonnement d'un modèle).  

Sur les 12 datasets, j'ai choisi:    
- [FinQA](https://arxiv.org/pdf/2109.00122) : basé sur des rapports financiers d'entreprise rendus publics. Des experts financiers étaient chargés de la création des questions et réponses.
- [ExpertQA](https://arxiv.org/pdf/2309.07852) : basé sur des recherches dans 32 domaines précis (médecine, juridique, histoire...). Les documents proviennent de pages web et des bases de données académiques ou professionnelles
- [TAT-QA](https://arxiv.org/pdf/2105.07624): basé également sur des rapports financiers réels.   

Ces datasets originaux ne sont pas prévus pour évaluer les citations. Les auteurs de RAGBench ont introduit des documents distracteurs avec un retriever. Une double annotation : modèle + humain a été utilisée pour confirmer les documents pertinents.  





RAGBench introduit une évaluation fine des documents : au niveau de la phrase. Pour chaque phrase de chaque chunk, ils indiquent si elle est pertinente pour répondre à la question. Pour le moment j'ai choisi de rester sur une évaluation au niveau du document : dès lors qu'un de ses phrases est jugée pertinente, il est ajouté au gold_ids.

In [1]:
import re

def custom_ragbench(raw_dataset):
    simplified_data = []

    for item in raw_dataset:
        # creation ideale de nos contextes
        contexts = [{"id": i, "text": doc_text} for i, doc_text in enumerate(item['documents'])]

        # recuperation des gold ids : des lors qu'une phrase d'un doc est dans la variable relevant sentences, on accepte ce doc en gold id
        gold_ids_set = set()
        for key in item.get('all_relevant_sentence_keys', []):
            match = re.match(r"(\d+)", key)
            if match:
                gold_ids_set.add(int(match.group(1)))

        # crate dataframe
        simplified_data.append({
            "id": item['id'],
            "source": item.get('dataset_name', 'ragbench_unknown'), # on ajoute cette colonne pour identifier la source du dataset
            "question": item['question'],
            "contexts": contexts,
            "gold_ids": sorted(list(gold_ids_set)),
            "answer": item['response']
        })

    return pd.DataFrame(simplified_data)




In [2]:
ds_finqa = load_dataset("galileo-ai/ragbench", "finqa", split="train[:2000]")
ds_expertqa = load_dataset("galileo-ai/ragbench", "expertqa", split="train[:2000]")
ds_tatqa = load_dataset("galileo-ai/ragbench", "tatqa", split="train[:2000]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

finqa/train-00000-of-00001.parquet:   0%|          | 0.00/61.1M [00:00<?, ?B/s]

finqa/validation-00000-of-00001.parquet:   0%|          | 0.00/5.94M [00:00<?, ?B/s]

finqa/test-00000-of-00001.parquet:   0%|          | 0.00/8.94M [00:00<?, ?B/s]

...

tatqa/validation-00000-of-00001.parquet:   0%|          | 0.00/4.84M [00:00<?, ?B/s]

tatqa/test-00000-of-00001.parquet:   0%|          | 0.00/4.75M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26430 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3336 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3338 [00:00<?, ? examples/s]

In [3]:
df_finqa = custom_ragbench(ds_finqa)
df_expertqa = custom_ragbench(ds_expertqa)
df_tatqa = custom_ragbench(ds_tatqa)

df_combined = pd.concat([df_finqa, df_expertqa, df_tatqa], ignore_index=True)

# random shuffle pour mélanger l apparition des 3 datasets
df_ragbench = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)


In [7]:
df_ragbench_1000 = df_ragbench.sample(n=1000, random_state=42)
df_ragbench_1000.to_parquet(f"{path}/data/ragbench2.parquet", index=False)

In [ ]:
df_ragbench.head(20)

,id,source,question,contexts,gold_ids,answer
0,finqa_test_1609,tatqa_test,"How is inventories, including wireless devices...","[{'id': 0, 'text': 'ACCOUNTING POLICY [["""", ""A...",[1],"Inventories, including wireless devices and me..."
1,finqa_test_1538,tatqa_test,What is the total fair value of money market f...,"[{'id': 0, 'text': 'NOTE 2. FAIR VALUE OF FINA...",[0],"According to the context provided, the total f..."
2,finqa_test_1341,tatqa_test,What is the excluded potential common shares f...,"[{'id': 0, 'text': '3. EARNINGS PER SHARE [[""(...",[5],"According to the information provided, the dil..."
3,finqa_6544,finqa_test,what percentage of total contractual obligatio...,"[{'id': 0, 'text': 'repurchase programs . we u...","[1, 2]",To calculate the percentage of total contractu...
4,finqa_test_1068,tatqa_test,"In 2019, what is the percentage constitution o...","[{'id': 0, 'text': '13. SHARE-BASED EMPLOYEE C...",[0],To find the percentage constitution of nonvest...
5,finqa_test_1113,tatqa_test,What drove the profitability in 2019?,"[{'id': 0, 'text': 'The provisions (benefits) ...",[2],"Based on the information provided, the profita..."
6,finqa_7353,finqa_test,what was the cumulative rent expense from 2004...,"[{'id': 0, 'text': 'unconditional purchase obl...",[2],According to the information provided:\n\nRent...
7,finqa_test_461,tatqa_test,How many years did net financed service contra...,"[{'id': 0, 'text': 'Financing Receivables and ...",[0],"Net financed service contracts exceeded $2,000..."
8,finqa_test_107,tatqa_test,What was the percentage change in Interest exp...,"[{'id': 0, 'text': 'American Tower Corporation...",[0],The Interest expense for American Tower Corpor...
9,finqa_test_921,tatqa_test,What was the pre-tax curtailment gain in 2018?,"[{'id': 0, 'text': 'The following table summar...",[3],"According to the given context, the pre-tax cu..."
